# Extração de Embeddings

**Arquitetura:**
```
produto ──► RoBERTa  ──► ℝⁿ (768) ─┐
        ──► DINOv2   ──► ℝᵐ (768) ──┼──► concat ──► MLP ──► ℝ⁴
        ──► num+cat  ──► ℝᵏ        ─┘
```

**Estratégia em 2 etapas para eficiência:**
1. Pré-computar e salvar todos os embeddings em disco (roda 1× na GPU)
2. Treinar a MLP apenas com os embeddings salvos (rápido, iterável)


## 1. Imports e configuração

In [1]:
import os
import json
import warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, AutoImageProcessor
from sklearn.preprocessing import LabelEncoder
from PIL import Image
import requests
from io import BytesIO
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
from sklearn.preprocessing import StandardScaler


warnings.filterwarnings("ignore")

# ── Reprodutibilidade ──────────────────────────────────────────────────────────
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")


Device: cuda


In [4]:
df = pd.read_csv("../data/cubagem_40k_amazon.csv")

targets = [
    "length_cm",   # length >= width >= height
    "width_cm",
    "height_cm",
    "weight_g",
]

## 2. Carregamento e preparação dos dados

In [5]:
df = df.dropna(subset=targets + ["title", "image_url"])
df_train = df[df["split"] == "train"].reset_index(drop=True)
df_val   = df[df["split"] == "val"  ].reset_index(drop=True)

## 3. Download das imagens de produto

In [6]:
def baixar_imagem(row, path="../images"):
    """Baixa a imagem de um produto e salva em disco. Retorna (asin, ok)."""
    asin = row["asin"]
    path = os.path.join(path, f"{asin}.jpg")
    if os.path.exists(path):
        return asin, True
    try:
        resp = requests.get(row["image_url"], timeout=10)
        resp.raise_for_status()
        img = Image.open(BytesIO(resp.content)).convert("RGB")
        img.save(path, "JPEG")
        return asin, True
    except Exception:
        return asin, False


def baixar_imagens_paralelo(df_part, n_workers=8):
    falhas = []
    rows = df_part.to_dict("records")
    with ThreadPoolExecutor(max_workers=n_workers) as ex:
        futures = {ex.submit(baixar_imagem, r): r["asin"] for r in rows}
        for fut in tqdm(as_completed(futures), total=len(futures), desc="Download"):
            asin, ok = fut.result()
            if not ok:
                falhas.append(asin)
    print(f"✔ Baixadas | ✘ Falhas: {len(falhas)}")
    return set(falhas)

In [7]:
# Baixa imagens
falhas = baixar_imagens_paralelo(df, 8)

Download: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 31986/31986 [16:11<00:00, 32.94it/s]


✔ Baixadas | ✘ Falhas: 36


Download: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 7999/7999 [03:35<00:00, 37.08it/s]

✔ Baixadas | ✘ Falhas: 4


## 4. Extração de embeddings de texto (RoBERTa — mean pooling)

In [17]:
def montar_texto(row):
    """Concatena título + categoria + início da descrição."""
    titulo   = str(row.get("title", ""))
    cat      = str(row.get("source_category", ""))
    desc     = str(row.get("description", ""))[:300]
    return f"{titulo} [SEP] {cat} [SEP] {desc}"

def mean_pool(last_hidden, attention_mask):
    """Pooling pela média dos tokens (ignora padding)."""
    mask = attention_mask.unsqueeze(-1).float()
    return (last_hidden * mask).sum(1) / mask.sum(1).clamp(min=1e-9)

def extrair_embeddings_texto(df_part, tokenizer, base_model):
    path_saida = os.path.join("../embeddings", f"text_embeddings.npz")
 
    if os.path.exists(path_saida):
        print(f"Embeddings de texto já existem. Carregando.")
        data = np.load(path_saida, allow_pickle=True)
        return data["embeddings"], data["asins"]
 
    textos = [montar_texto(r) for r in df_part.to_dict("records")]
    asins  = df_part["asin"].values  # same order as `textos`, since both come from df_part
    embs   = []
    base_model.eval()
    for i in tqdm(range(0, len(textos), 64), desc=f"RoBERTa"):
        batch_txt = textos[i : i + 64]
        enc = tokenizer(
            batch_txt,
            padding=True,
            truncation=True,
            max_length=256,
            return_tensors="pt",
        ).to(device)
        with torch.no_grad():
            out  = base_model(**enc)
            pool = mean_pool(out.last_hidden_state, enc["attention_mask"])
            embs.append(pool.cpu().numpy())
    embs_np = np.concatenate(embs, axis=0)
 
    np.savez(path_saida, embeddings=embs_np, asins=asins)
 
    return embs_np, asins

In [18]:
tokenizer_rob  = AutoTokenizer.from_pretrained("hyp1231/blair-roberta-base")
roberta_model  = AutoModel.from_pretrained("hyp1231/blair-roberta-base").to(device)

for p in roberta_model.parameters():   # congela para extração
    p.requires_grad = False

extrair_embeddings_texto(df, tokenizer_rob, roberta_model)

# Libera memória GPU
del roberta_model
torch.cuda.empty_cache()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RoBERTa: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 625/625 [02:46<00:00,  3.75it/s]


## 5. Extração de embeddings de imagem (DINOv2)

In [19]:
def carregar_imagem(asin):
    path = os.path.join("../images", f"{asin}.jpg")
    try:
        return Image.open(path).convert("RGB")
    except Exception:
        return Image.new("RGB", (224, 224), color=128)


def extrair_embeddings_imagem(df_part, processor, dino_model):
    path_saida = os.path.join("../embeddings", f"imagem_embeddings.npz")

    if os.path.exists(path_saida):
        print(f"Embeddings de imagem já existem. Carregando.")
        data = np.load(path_saida, allow_pickle=True)
        return data["embeddings"], data["asins"]

    asins = df_part["asin"].tolist()
    embs  = []
    dino_model.eval()
    for i in tqdm(range(0, len(asins), 32), desc=f"DINOv2"):
        batch_asins = asins[i : i + 32]
        imgs = [carregar_imagem(a) for a in batch_asins]
        enc = processor(images=imgs, return_tensors="pt").to(device)
        with torch.no_grad():
            out  = dino_model(**enc)
            # CLS token — representa o embedding global da imagem
            pool = out.last_hidden_state[:, 0, :]
            embs.append(pool.cpu().numpy())
    embs_np = np.concatenate(embs, axis=0)

    np.savez(path_saida, embeddings=embs_np, asins=np.asarray(asins))

    return embs_np, asins

In [20]:
dino_processor = AutoImageProcessor.from_pretrained("facebook/dinov2-base")
dino_model     = AutoModel.from_pretrained("facebook/dinov2-base").to(device)

for p in dino_model.parameters():
    p.requires_grad = False

emb_image = extrair_embeddings_imagem(df, dino_processor, dino_model,)

del dino_model
torch.cuda.empty_cache()

preprocessor_config.json:   0%|          | 0.00/436 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/548 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

DINOv2: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1250/1250 [09:35<00:00,  2.17it/s]
